# nested_loops
Nested loops appear whenever work depends on combinations, grids, matrix-like data, grouped records, or comparisons across sets. They are simple to write, but they are also where accidental O(n^2) behavior and unreadable control flow start to show up in real systems.

## 1. Problem First
Why does this matter in real systems?
- Comparing every request against every rule can explode runtime.
- Processing tables, CSV rows and columns, or matrices naturally creates nested iteration.
- Duplicate detection and pairwise matching often start as nested loops.

In [1]:
rows = [[1, 2], [3, 4]]
for row in rows:
    for value in row:
        print(value)

1
2
3
4


## 2. Minimal Concept
Core syntax:
- A loop can contain another loop.
- The inner loop runs fully for each outer-loop iteration.
- Total work is usually outer_count × inner_count.

## 3. Mental Model
How Python actually behaves
Nested loops create a product of paths. For each value from the outer iterable, Python restarts the inner loop from the beginning. That means the inner work repeats many times, which is correct for pairwise tasks but wasteful if the repeated work was not intended.

In [2]:
services = ["api", "worker"]
regions = ["us", "eu", "apac"]
for service in services:
    for region in regions:
        print(service, region)

api us
api eu
api apac
worker us
worker eu
worker apac


## 4. Internal Mechanics
A nested loop is just one iterator-driving block inside another. Python will repeatedly create or resume the inner iterator for each outer item. The important proof is not mystery behavior, but how quickly total iterations multiply.

In [3]:
import dis

def pairs(left, right):
    for a in left:
        for b in right:
            print(a, b)

dis.dis(pairs)
pairs([1, 2], ["x", "y"])

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (left)
              4 GET_ITER
        >>    6 FOR_ITER                21 (to 52)
             10 STORE_FAST               2 (a)

  5          12 LOAD_FAST                1 (right)
             14 GET_ITER
        >>   16 FOR_ITER                14 (to 48)
             20 STORE_FAST               3 (b)

  6          22 LOAD_GLOBAL              1 (NULL + print)
             32 LOAD_FAST                2 (a)
             34 LOAD_FAST                3 (b)
             36 CALL                     2
             44 POP_TOP
             46 JUMP_BACKWARD           16 (to 16)

  5     >>   48 END_FOR
             50 JUMP_BACKWARD           23 (to 6)

  4     >>   52 END_FOR
             54 RETURN_CONST             0 (None)
1 x
1 y
2 x
2 y


## 5. Edge Cases
Where it breaks:
- The same expensive inner work runs for every outer item.
- State from the inner loop may leak into later logic in confusing ways.
- A nested loop can hide accidental duplicate processing.
- Early exits become harder to reason about when both loops matter.

In [4]:
users = ["alice", "bob", "alice"]
duplicates = []
for i in range(len(users)):
    for j in range(len(users)):
        if i != j and users[i] == users[j]:
            duplicates.append(users[i])
print(duplicates)

['alice', 'alice']


## 6. Performance Thinking
- If the outer loop is O(n) and the inner loop is O(m), total work is O(n*m).
- If both dimensions scale together, nested loops often become O(n^2).
- That may be acceptable for tiny inputs and unacceptable for production-scale datasets.
- Precomputed lookups like sets or dictionaries often replace an inner loop and reduce cost dramatically.

## 7. Real Use Case
A security tool checks each incoming hostname against a list of blocked patterns. The straightforward solution is often a nested loop.

In [5]:
hosts = ["api.internal", "evil.example", "db.internal"]
blocked_patterns = ["evil", "malware"]
matches = []
for host in hosts:
    for pattern in blocked_patterns:
        if pattern in host:
            matches.append(host)
print(matches)

['evil.example']


## 8. Anti-Pattern
What beginners do wrong:
- Reach for nested loops before asking whether a lookup structure would work.
- Duplicate comparisons by checking every pair twice.
- Mix too much branching and mutation inside both loop levels.

In [6]:
numbers = [1, 2, 3, 2]
for i in range(len(numbers)):
    for j in range(len(numbers)):
        if i != j and numbers[i] == numbers[j]:
            print("duplicate", numbers[i])

duplicate 2
duplicate 2


## 9. Interview Signals
Questions FAANG asks:
- When is O(n^2) from nested loops acceptable?
- How would you replace an inner loop with a set or dictionary?
- Why do duplicate-detection problems often start with nested loops?
- How do you reason about correctness and performance together here?

## 10. Exercise (Non-trivial)
You are building a reconciliation job that compares two datasets: records from an API and records from a warehouse export. Start with a nested-loop solution, then redesign it so lookups are faster, duplicates are handled explicitly, and mismatch reporting remains readable.

In [7]:
def find_missing(api_records, warehouse_records):
    missing = []
    for api_record in api_records:
        found = False
        for warehouse_record in warehouse_records:
            if api_record["id"] == warehouse_record["id"]:
                found = True
        if not found:
            missing.append(api_record)
    return missing

# Task:
# 1. Analyze the time complexity.
# 2. Refactor with a lookup structure.
# 3. Keep mismatch reporting easy to understand.